# reader

> parsing lisp text into an AST

In [ ]:
#| default_exp reader

## Atoms and Literals

In [ ]:
import ast, re
from fastcore.basics import basic_repr, store_attr, first, last
from compact.types import *

In [ ]:
#| hide
ast.literal_eval("123"), ast.literal_eval('"hello world"'), ast.literal_eval("2+3j")

(123, 'hello world', (2+3j))

In [ ]:
def Atom(s):
    "parse atoms, default this is a symbol"
    if s == "#t": return True
    if s == "#f": return False
    try: return ast.literal_eval(s)
    except Exception: return Symbol(s)


In [ ]:
Atom("123"), Atom('"hello world"'), Atom("2+3j"), Atom("lambda")

(123, 'hello world', (2+3j), compact.types.Symbol(s='lambda'))

In [ ]:
Atom("+") == Atom("+")

True

In [ ]:
Atom("+") == Atom('"+"')

False

## Reader: Lexer and Parser
parse strings into lisp expressions

In [ ]:
STR = r'"(?:\\.|[^"\\])*"'  # string with spaces etc.
COMMENT = r';[^\n]*'

DOT = r'\.'

ATOM = r'''[^\s()`',;]+'''

UNQ_SPL = ',@'              # unquote splice
PAREN = '[()]'
QUOTE = "[`',]"

# order matters `,@` needs to show up before the single character `,`
TOKEN_RE = "|".join([STR, COMMENT, DOT, UNQ_SPL, PAREN, QUOTE, ATOM])

def lexer(s): return [t for t in re.findall(TOKEN_RE, s) if not t.startswith(';')]

In [ ]:
lexer("( + 1 2 )")

['(', '+', '1', '2', ')']

In [ ]:
lexer("'( + 1 2 )")

["'", '(', '+', '1', '2', ')']

In [ ]:
lexer("`( + 1 2 )")

['`', '(', '+', '1', '2', ')']

In [ ]:
lexer('`(+ ,@xs 2)')

['`', '(', '+', ',@', 'xs', '2', ')']

In [ ]:
lexer("(+ 1 2)")

['(', '+', '1', '2', ')']

In [ ]:
lexer("'(1 2 3)")

["'", '(', '1', '2', '3', ')']

In [ ]:
def parser(toks):
    if not toks: raise SyntaxError("malformed list: unexpected EOF")
    
    t = toks.pop(0)
    if t == ')': raise SyntaxError("malformed list: unexpected )")
    if t == '(':
        sl = []
        while toks and toks[0] != ')': sl.append(parser(toks))
        if not toks: raise SyntaxError("malformed list: missing )")
        toks.pop(0)
        return sl
    sugar = {
        "'": "quote",
        "`": "quasiquote",
        ",": "unquote",
        ",@": "unquote-splicing",
    }
    if t in sugar: return [Symbol(sugar[t]), parser(toks)]

    return Atom(t)

In [ ]:
parser(lexer('`(+ ,@xs 2)'))

[compact.types.Symbol(s='quasiquote'),
 [compact.types.Symbol(s='+'),
  [compact.types.Symbol(s='unquote-splicing'), compact.types.Symbol(s='xs')],
  2]]

In [ ]:
def parse(s): 
    toks = lexer(s)
    r = parser(toks)
    if not toks: return r
    raise SyntaxError("unexpected tokens after expression")

In [ ]:
parse('`(+ ,@xs 2)')

[compact.types.Symbol(s='quasiquote'),
 [compact.types.Symbol(s='+'),
  [compact.types.Symbol(s='unquote-splicing'), compact.types.Symbol(s='xs')],
  2]]

In [ ]:
def parse_all(s):
    toks = lexer(s)
    exprs = []
    while toks: exprs.append(parser(toks))
    return exprs

In [ ]:
parse_all("""
(define x 10)
(+ x 5)
""")

[[compact.types.Symbol(s='define'), compact.types.Symbol(s='x'), 10],
 [compact.types.Symbol(s='+'), compact.types.Symbol(s='x'), 5]]

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()